In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb

In [25]:
df = pd.read_parquet("d:/FIAP/Engenharia de Machine Learning/Fase 3/tech_challenge_3/data/processed/flights_processed.parquet")


KeyboardInterrupt



In [ ]:
pd.set_option('display.max_columns', None) 
df.head()

In [ ]:
# Suponha que df tenha: TAIL_NUMBER, DATE, SCHEDULED_DEPARTURE, DEPARTURE_DELAY
# Ordenar por aeronave e horário
df = df.sort_values(by=['TAIL_NUMBER', 'DATE', 'SCHEDULED_DEPARTURE'])

# Definir atraso do voo anterior
df['PREVIOUS_DELAY'] = df.groupby('TAIL_NUMBER')['DEPARTURE_DELAY'].shift(1)

# Criar flag de atraso do voo anterior (ex: >= 15 minutos)
df['IS_PREVIOUS_FLIGHT_DELAYED'] = (df['PREVIOUS_DELAY'] >= 15).astype(int)

# Preencher valores NA (primeiro voo de cada aeronave) com 0
df['IS_PREVIOUS_FLIGHT_DELAYED'] = df['IS_PREVIOUS_FLIGHT_DELAYED'].fillna(0)

In [ ]:
def create_top_n_feature(
    df, 
    column, 
    target="IS_DELAYED", 
    top_n=10, 
    new_column_name=None,
    drop_original=True
):
    """
    Cria uma coluna categórica "Top N + Other" baseada na proporção de ocorrências
    do target para cada categoria da coluna especificada, compatível com colunas
    categóricas já existentes.

    Parâmetros:
    ----------
    df : pd.DataFrame
        DataFrame original
    column : str
        Nome da coluna que você quer agrupar (ex: 'ORIGIN_AIRPORT')
    target : str, default='IS_DELAYED'
        Coluna alvo usada para calcular o ranking (ex: considerar apenas atrasos)
    top_n : int, default=10
        Número de categorias mais frequentes a manter
    new_column_name : str, default=None
        Nome da nova coluna. Se None, será "{column}_TOP{top_n}"
    drop_original : bool, default=False
        Se True, remove a coluna original após criar a coluna Top N

    Retorna:
    -------
    pd.DataFrame com a nova coluna adicionada
    """
    
    if new_column_name is None:
        new_column_name = f"{column}_TOP{top_n}"

    # Filtrar apenas os casos relevantes do target (ex: voos atrasados)
    filtered = df[df[target] == 1]

    # Contar ocorrências por categoria
    counts = filtered[column].value_counts()

    # Selecionar top N
    top_categories = counts.head(top_n).index

    # Se a coluna for Categorical, adiciona "Other" antes
    if pd.api.types.is_categorical_dtype(df[column]):
        df[column] = df[column].cat.add_categories("Other")

    # Criar coluna nova: mantém top N, resto vira "Other"
    df[new_column_name] = df[column].where(df[column].isin(top_categories), "Other")

    # Remover coluna original, se solicitado
    if drop_original:
        df = df.drop(columns=[column])
    
    return df

In [ ]:
# Top 10 aeroportos de origem
df = create_top_n_feature(df, "ORIGIN_AIRPORT", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "DESTINATION_AIRPORT", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "CITY", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "STATE", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "TAIL_NUMBER", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "AIRLINE", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "FLIGHT_NUMBER", target="IS_DELAYED", top_n=10)


In [ ]:
def hhmm_to_minutes(hhmm):
    hhmm = int(hhmm)            # garante que é inteiro
    hours = hhmm // 100          # divide para pegar as horas
    minutes = hhmm % 100         # pega os minutos restantes
    return hours*60 + minutes    # total em minutos

# Exemplo com pandas
df['SCHEDULED_DEPARTURE_MIN'] = df['SCHEDULED_DEPARTURE'].apply(hhmm_to_minutes)

In [ ]:
df.head()

In [ ]:
remove = ["YEAR", "DAY","LATITUDE", "LONGITUDE", "DATE", "AIRPORT", "AIRLINE_CODE", "SCHEDULED_DEPARTURE", "HOUR"]

df = df.drop(columns=remove)

In [ ]:
categorical_features = [ "MONTH", "DAY_OF_WEEK"]
df = pd.get_dummies(df, columns=categorical_features)

In [ ]:
categorical_features = ["ORIGIN_AIRPORT_TOP10","DESTINATION_AIRPORT_TOP10"]
df = pd.get_dummies(df, columns=categorical_features)

In [ ]:
categorical_features = ["CITY_TOP10", "STATE_TOP10"]
df = pd.get_dummies(df, columns=categorical_features)

In [ ]:
remove = ["TAIL_NUMBER_TOP10"]

df = df.drop(columns=remove)

In [ ]:
categorical_features = ["AIRLINE_TOP10", "FLIGHT_NUMBER_TOP10"]
df = pd.get_dummies(df, columns=categorical_features)

In [ ]:
from sklearn.preprocessing import StandardScaler
continuous_features = ["DEPARTURE_DELAY", "SCHEDULED_TIME", "DISTANCE", "SCHEDULED_ARRIVAL"]
scaler = StandardScaler()
df[continuous_features] = scaler.fit_transform(df[continuous_features])

In [ ]:
df.head()

In [ ]:
#separa features e target 
X = df.drop(columns=["IS_DELAYED", "DEPARTURE_DELAY"])
y = df["IS_DELAYED"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
X_train.head()

In [ ]:
#from imblearn.under_sampling import RandomUnderSampler
#from collections import Counter
#rus = RandomUnderSampler(random_state=42)
#X_res, y_res = rus.fit_resample(X, y)

#print(Counter(y))
#print(Counter(y_res))

In [ ]:
# Ajustando para desbalanceamento
scale = sum(y_train==0) / sum(y_train==1)  # razão de classes
model = xgb.XGBClassifier(
    scale_pos_weight=scale,
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=42
)

# Treinar
model.fit(X_train, y_train)

# Previsão
y_prob = model.predict_proba(X_test)[:,1]

# Ajustando threshold manual ou usando PR curve depois
threshold = 0.533
y_pred = (y_prob >= threshold).astype(int)

# Avaliar
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * (precision * recall) / (precision + recall)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print("Melhor threshold:", best_threshold)

In [ ]:
# Features que têm maior impacto
important_features = [
    "DAY_OF_WEEK",
    "SCHEDULED_ARRIVAL",
    "SCHEDULED_DEPARTURE",
    "FLIGHT_NUMBER",
    "SCHEDULED_TIME",
    "DISTANCE",
    "HOUR",
    "AIRLINE_CODE",
    "DESTINATION_AIRPORT_TOP10",
    "STATE_TOP10",
    "CITY_TOP10",
    "ORIGIN_AIRPORT_TOP10"
]

In [ ]:
from sklearn.svm import SVC

# Para SVM precisamos de menos amostras ou subsample, porque é lento
X_sample = X_train_scaled.sample(n=50000, random_state=42)
y_sample = y_train.loc[X_sample.index]

svm = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)
svm.fit(X_sample, y_sample)

y_prob = svm.predict_proba(X_test_scaled)[:,1]
y_pred = (y_prob >= threshold).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Normalizar variáveis contínuas
cont_features = ["DISTANCE", "SCHEDULED_DEPARTURE_MIN", "SCHEDULED_TIME", "SCHEDULED_ARRIVAL"]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[cont_features] = scaler.fit_transform(X_train[cont_features])
X_test_scaled[cont_features] = scaler.transform(X_test[cont_features])

# Target = tempo de atraso real
y_train_reg = y_train_delay  # por exemplo, DEPARTURE_DELAY
y_test_reg = y_test_delay

# Treinar modelo
reg = LinearRegression()
reg.fit(X_train_scaled, y_train_reg)

# Previsão
y_pred_reg = reg.predict(X_test_scaled)

# Avaliar
mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
print("MSE:", mse)
print("R²:", r2)
